# 01 — Extraction

**Project:** Identifying high-risk road conditions and geographic hotspots to reduce traffic accident severity across the United States.

**Goal of this notebook:** Load the raw US Accidents dataset, inspect its structure, and confirm it meets the Capstone 2 dataset requirements (≥ 10,000 rows, ≥ 8 analytical columns, realistic data-quality issues).

**Source:** [US Accidents (Feb 2016 – Mar 2023)](https://www.kaggle.com/datasets/sobhanmoosavi/us-accidents) — Moosavi, S. et al. Sourced via live traffic APIs across the contiguous United States.

**Before running:** execute `python data/raw/combine.py` once to reassemble the full CSV from the two split parts.

In [1]:
import subprocess
import sys
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().resolve().name == 'notebooks' else Path.cwd().resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from scripts.etl_pipeline import normalize_columns

RAW_PATH = PROJECT_ROOT / 'data' / 'raw' / 'US_Accidents_March23.csv'
INSPECTION_ROWS = 100_000

print('Raw file exists:', RAW_PATH.exists())
print('Raw file path  :', RAW_PATH)
print(f'Inspection rows: {INSPECTION_ROWS:,}')

Raw file exists: True
Raw file path  : /Users/aryankinha/Downloads/B_G14_DVACapstone-main/data/raw/US_Accidents_March23.csv
Inspection rows: 100,000


## Load an inspection sample

The full dataset is ~2.8 GB / ~7.7M rows. For notebook responsiveness, this extraction notebook inspects the first `100,000` rows and separately records the full file row count using `wc -l`. Cleaning in notebook 02 uses a representative random sample for Tableau.

In [2]:
line_count = int(subprocess.check_output(['wc', '-l', str(RAW_PATH)]).split()[0])
raw_row_count = line_count - 1

df_raw = pd.read_csv(RAW_PATH, low_memory=False, nrows=INSPECTION_ROWS)
print(f'Full rows      : {raw_row_count:,}')
print(f'Inspection rows: {len(df_raw):,}')
print(f'Columns        : {df_raw.shape[1]}')
print(f'Sample memory  : {df_raw.memory_usage(deep=True).sum() / 1e6:.2f} MB')

Full rows      : 7,728,394
Inspection rows: 100,000
Columns        : 46
Sample memory  : 131.31 MB


In [3]:
df_raw.head()

,ID,Source,Severity,Start_Time,End_Time,Start_Lat,Start_Lng,End_Lat,End_Lng,Distance(mi),...,Roundabout,Station,Stop,Traffic_Calming,Traffic_Signal,Turning_Loop,Sunrise_Sunset,Civil_Twilight,Nautical_Twilight,Astronomical_Twilight
0,A-1,Source2,3,2016-02-08 05:46:00,2016-02-08 11:00:00,39.865147,-84.058723,NaN,NaN,0.01,...,False,False,False,False,False,False,Night,Night,Night,Night
1,A-2,Source2,2,2016-02-08 06:07:59,2016-02-08 06:37:59,39.928059,-82.831184,NaN,NaN,0.01,...,False,False,False,False,False,False,Night,Night,Night,Day
2,A-3,Source2,2,2016-02-08 06:49:27,2016-02-08 07:19:27,39.063148,-84.032608,NaN,NaN,0.01,...,False,False,False,False,True,False,Night,Night,Day,Day
3,A-4,Source2,3,2016-02-08 07:23:34,2016-02-08 07:53:34,39.747753,-84.205582,NaN,NaN,0.01,...,False,False,False,False,False,False,Night,Day,Day,Day
4,A-5,Source2,2,2016-02-08 07:39:07,2016-02-08 08:09:07,39.627781,-84.188354,NaN,NaN,0.01,...,False,False,False,False,True,False,Day,Day,Day,Day


In [4]:
df_raw.info(verbose=True, show_counts=True)

<class 'pandas.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 46 columns):
 #   Column                 Non-Null Count   Dtype  
---  ------                 --------------   -----  
 0   ID                     100000 non-null  str    
 1   Source                 100000 non-null  str    
 2   Severity               100000 non-null  int64  
 3   Start_Time             100000 non-null  str    
 4   End_Time               100000 non-null  str    
 5   Start_Lat              100000 non-null  float64
 6   Start_Lng              100000 non-null  float64
 7   End_Lat                0 non-null       float64
 8   End_Lng                0 non-null       float64
 9   Distance(mi)           100000 non-null  float64
 10  Description            100000 non-null  str    
 11  Street                 100000 non-null  str    
 12  City                   99999 non-null   str    
 13  County                 100000 non-null  str    
 14  State                  100000 non-null  str    


## Column inventory

Normalize column names to snake_case so downstream notebooks can rely on stable identifiers.

In [5]:
df = normalize_columns(df_raw)
print('Normalized column names:')
for c in df.columns:
    print(f'  - {c}')

Normalized column names:
  - id
  - source
  - severity
  - start_time
  - end_time
  - start_lat
  - start_lng
  - end_lat
  - end_lng
  - distance_mi
  - description
  - street
  - city
  - county
  - state
  - zipcode
  - country
  - timezone
  - airport_code
  - weather_timestamp
  - temperature_f
  - wind_chill_f
  - humidity
  - pressure_in
  - visibility_mi
  - wind_direction
  - wind_speed_mph
  - precipitation_in
  - weather_condition
  - amenity
  - bump
  - crossing
  - give_way
  - junction
  - no_exit
  - railway
  - roundabout
  - station
  - stop
  - traffic_calming
  - traffic_signal
  - turning_loop
  - sunrise_sunset
  - civil_twilight
  - nautical_twilight
  - astronomical_twilight


## Missing-value audit

Which columns have missing data? Percentages drive the imputation strategy in notebook 02.

In [6]:
missing = pd.DataFrame({
    'null_count': df.isna().sum(),
    'null_pct'  : (df.isna().mean() * 100).round(2),
    'dtype'     : df.dtypes.astype(str),
})
missing = missing.sort_values('null_pct', ascending=False)
missing

,null_count,null_pct,dtype
end_lat,100000,100.00,float64
end_lng,100000,100.00,float64
wind_chill_f,95678,95.68,float64
precipitation_in,92632,92.63,float64
wind_speed_mph,23820,23.82,float64
humidity,1856,1.86,float64
visibility_mi,1846,1.85,float64
weather_condition,1604,1.60,str
temperature_f,1591,1.59,float64
pressure_in,1292,1.29,float64


## Key descriptive statistics

In [7]:
df.describe(include='all', ).T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
id,100000,100000,A-1,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
source,100000,2,Source2,99950,NaN,NaN,NaN,NaN,NaN,NaN,NaN
severity,100000.0,NaN,NaN,NaN,2.44812,0.499931,1.0,2.0,2.0,3.0,4.0
start_time,100000,98896,2016-04-10 08:59:26,35,NaN,NaN,NaN,NaN,NaN,NaN,NaN
end_time,100000,96724,2016-10-14 19:50:00,24,NaN,NaN,NaN,NaN,NaN,NaN,NaN
start_lat,100000.0,NaN,NaN,NaN,35.630369,2.101259,32.542587,33.957775,34.168579,37.776682,41.428753
start_lng,100000.0,NaN,NaN,NaN,-119.322756,3.663525,-123.813927,-121.828468,-118.384232,-117.916023,-81.550728
end_lat,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
end_lng,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
distance_mi,100000.0,NaN,NaN,NaN,0.011345,0.262244,0.0,0.0,0.0,0.01,51.13


In [8]:
# Time range and severity distribution — the two headline dimensions of the study.
start_time = pd.to_datetime(df['start_time'], errors='coerce')
print('Date range :', start_time.min(), '→', start_time.max())
print('Unique states:', df['state'].nunique())
print('Unique cities:', df['city'].nunique())
print('\nSeverity distribution (%):')
print((df['severity'].value_counts(normalize=True).sort_index() * 100).round(2))

Date range : 2016-02-08 05:46:00 → 2017-01-26 20:07:03
Unique states: 3
Unique cities: 785

Severity distribution (%):
severity
1     0.10
2    55.02
3    44.84
4     0.03
Name: proportion, dtype: float64


## Capstone 2 dataset compliance check

| Requirement | Target | Actual |
|---|---|---|
| Row count | ≥ 10,000 | see below |
| Column count | ≥ 8 meaningful | see below |
| Realistic data-quality issues | missing values / inconsistent formats | see above |
| Format | tabular, row-level | ✓ |

In [9]:
assert raw_row_count >= 10_000, 'Dataset does not meet the minimum row requirement.'
assert df.shape[1] >= 8, 'Dataset does not meet the minimum column requirement.'
print('✓ Dataset meets Capstone 2 minimum size requirements.')
print(f'  Full rows      : {raw_row_count:,}')
print(f'  Columns        : {df.shape[1]}')
print(f'  Inspected rows : {len(df):,}')

✓ Dataset meets Capstone 2 minimum size requirements.
  Full rows      : 7,728,394
  Columns        : 46
  Inspected rows : 100,000


### Takeaways for cleaning (notebook 02)

- High-null columns like `end_lat`, `end_lng`, `number`, `precipitation_in`, `wind_chill_f` will be dropped or median-imputed.
- `start_time` / `end_time` require datetime parsing to derive `duration_min`, `hour`, `day_of_week`, `month`, `season`.
- `severity` is already integer 1–4 → map to readable labels and a binary high-severity flag.
- Boolean POI columns (traffic_signal, junction, crossing, …) need explicit casting to `bool`.
- Outliers exist in `distance_mi`; we will cap at the 99th percentile.

Proceed to **`02_cleaning.ipynb`**.